In [1]:
import folium
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

In [31]:
wards = {
    'Quan_7': {'latitude': 10.7342, 'longitude': 106.7218, 'type': 'residential'},
    'Quan_10': {'latitude': 10.7725, 'longitude': 106.6683, 'type': 'office'},
    'Tan_Binh': {'latitude': 10.8016, 'longitude': 106.6520, 'type': 'airport'}
}
data = []

for i in range (1000):
  ward_name = np.random.choice(list(wards.keys()))
  hour = np.random.randint(0, 24)
  ward_demand = np.random.randint(10, 50)
  is_weekend = np.random.choice([0, 1])
  if ward_name == 'Quan_7' and (hour in [7,8,18,19,20]) and not is_weekend:
    demand = ward_demand * 2.5
  elif ward_name == 'Quan_10' and (hour in [7,8,17,18]):
    demand = ward_demand * 3
  elif ward_name == 'Tan_Binh' and (7<=hour<=22):
    demand = ward_demand * 2
  else:
    demand = ward_demand * 1.5
  data.append([ward_name, hour, is_weekend, demand])

df = pd.DataFrame(data, columns=['Ward', 'Hour', 'Is_Weekend', 'Demand'])

In [32]:
df['Ward_code'] = df['Ward'].astype('category').cat.codes
X = df[['Ward_code', 'Hour', 'Is_Weekend']]
y = df['Demand']

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)
future_hour, future_weekend= 17, 0

In [34]:
predictions = {}
for ward_name in wards.keys():
  ward_code = df['Ward'].astype('category').cat.categories.get_loc(ward_name)
  pred_demand = model.predict([[ward_code, future_hour, future_weekend]])[0]
  predictions[ward_name] = pred_demand
  print(f"Khu vực {ward_name}: Dự kiến {pred_demand:.0f} cuốc xe")

Khu vực Quan_7: Dự kiến 39 cuốc xe
Khu vực Quan_10: Dự kiến 91 cuốc xe
Khu vực Tan_Binh: Dự kiến 61 cuốc xe


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [35]:
m = folium.Map(location=[10.7725, 106.69], zoom_start=13)
for ward, coords in wards.items():
  color = 'red' if predictions[ward] > 80 else 'blue'
  folium.CircleMarker(
      location=[coords['latitude'], coords['longitude']],
      radius=predictions[ward]/5,
      color=color, fill=True, fill_color=color, fill_opacity=0.5
  ).add_to(m)
m

Khi thay đổi biến future_hour (từ 17h thành 12h trưa hay 7h sáng), các vòng tròn trên bản đồ sẽ phình to hoặc thu nhỏ tùy theo khu vực.

Các giá trị quản trị cốt lõi:
*   Điều phối chủ động: Thay vì đợi khách hàng bật app lên gọi xe rồi mới bắt đầu tìm tài xế , AI giúp dự đoán trước thời gian cao điểm, từ đó tài xế ở các khu lân cận có thể chủ động chạy xe về khu vực cần đón theo nhu cầu, giúp khách rút ngắn thời gian chờ.
*   Định giá linh hoạt: Khi mô hình dự đoán nhu cầu tại khu vực xác định số cuốc > số tài xế khả dụng, thuật toán sẽ tự động kích hoạt nâng giá. Việc này vừa giúp giảm bớt những khách hàng không quá vội, đồng thời dùng tiền thưởng cao để thu hút thêm tài xế từ khu vực khác.